In [1]:
#CosineAnnealingLR を導入して scheduler.step() を追加

In [9]:
import os
import json
import numpy as np
from PIL import Image
from tqdm import tqdm
from sklearn.model_selection import train_test_split

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, Subset

# データセット
class ModeAndFeatureDataset(Dataset):
    def __init__(self, crop_root, annot_root, distance_json_path=None, max_items=None):
        self.items = []
        self.distances = {}
        if distance_json_path:
            with open(distance_json_path, encoding='utf-8') as f:
                self.distances = json.load(f)

        scene_ids = sorted(os.listdir(crop_root))
        for sid in scene_ids:
            if not sid.isdigit():
                continue
            crop_dir = os.path.join(crop_root, sid)
            annot_path = os.path.join(annot_root, f"{sid}.json")
            if not os.path.exists(annot_path): continue

            files = sorted([f for f in os.listdir(crop_dir) if f.endswith(".png")])
            if len(files) == 0: continue

            with open(annot_path, encoding='utf-8') as f:
                ann = json.load(f)

            seq = ann['sequence']
            min_len = min(len(files), len(seq))
            if min_len < 15: continue

            own_speeds = np.array([f['OwnSpeed'] / 3.6 for f in seq], dtype=np.float32)
            tgt_speeds = np.array([f['TgtSpeed_ref'] / 3.6 for f in seq], dtype=np.float32)
            angles = np.array([f['StrDeg'] for f in seq], dtype=np.float32)
            dists_all = [self.distances.get(sid, {}).get(str(i), 0.0) for i in range(min_len)]
            dists_all = np.array(dists_all, dtype=np.float32)

            def smooth(x, k):
                return np.convolve(x, np.ones(k)/k, mode='same')

            for i in range(min_len - 14):
                if max_items and len(self.items) >= max_items:
                    return

                d = dists_all[i:i+15]
                s = own_speeds[i:i+15]
                a = angles[i:i+15]
                t = tgt_speeds[i:i+15]

                d1 = np.gradient(d)
                d2 = np.gradient(d1)
                s1 = np.gradient(s)
                rel_acc = d2 - np.mean(d2)

                d_smooths = []
                for w in [3, 5, 7, 11]:
                    smoothed = smooth(d, w)[:15]
                    d_smooths.append(smoothed)
                    d_smooths.append(np.gradient(smoothed))

                img_paths = [os.path.join(crop_dir, files[j]) for j in range(i, i + 15)]
                modes = []
                for p in img_paths:
                    img = np.array(Image.open(p).convert("L")).flatten()
                    if img.size == 0:
                        img_mode = 0.0
                    else:
                        vals, counts = np.unique(img, return_counts=True)
                        img_mode = float(vals[np.argmax(counts)]) / 255.0
                    modes.append(img_mode)

                rel_speed = np.mean(t - s)

                feature = np.concatenate([modes, d, s, a, s1, d1, d2, rel_acc] + d_smooths)
                self.items.append((feature.astype(np.float32), rel_speed, sid))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        feature, tgt, sid = self.items[idx]
        return torch.tensor(feature), torch.tensor(tgt, dtype=torch.float32), sid


def collate_fn(batch):
    feats, tgts, sids = zip(*batch)
    return torch.stack(feats), torch.tensor(tgts), list(sids)

#モデル定義
class ExtendedFeatureModel(nn.Module):
    def __init__(self, in_dim=225):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(in_dim, 512), nn.BatchNorm1d(512), nn.ReLU(),
            nn.Linear(512, 256), nn.BatchNorm1d(256), nn.ReLU(),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64), nn.BatchNorm1d(64), nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.fc(x).squeeze(1)

#Training roop
def train_extended_model(dataset, save_path="420_3.pth"):
    scenes = sorted(list(set([item[-1] for item in dataset.items])))
    train_scenes, val_scenes = train_test_split(scenes, test_size=0.2, random_state=42)

    train_idx = [i for i, item in enumerate(dataset.items) if item[-1] in train_scenes]
    val_idx = [i for i, item in enumerate(dataset.items) if item[-1] in val_scenes]

    train_ds = Subset(dataset, train_idx[:8000])
    val_ds = Subset(dataset, val_idx[:2000])

    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, collate_fn=collate_fn)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = ExtendedFeatureModel(in_dim=train_ds[0][0].shape[0]).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-2)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30, eta_min=1e-5)
    criterion = nn.SmoothL1Loss()

    best_val_loss = float('inf')
    patience = 250
    patience_counter = 0

    for epoch in range(250):
        model.train()
        total_train_loss = 0
        for feats, tgts, _ in tqdm(train_loader, desc=f"[Train {epoch+1}]"):
            feats, tgts = feats.to(device), tgts.to(device)
            pred = model(feats)
            loss = criterion(pred, tgts)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_train_loss += loss.item() * feats.size(0)

        model.eval()
        total_val_loss = 0
        with torch.no_grad():
            for feats, tgts, _ in val_loader:
                feats, tgts = feats.to(device), tgts.to(device)
                pred = model(feats)
                loss = criterion(pred, tgts)
                total_val_loss += loss.item() * feats.size(0)

        train_loss = total_train_loss / len(train_ds)
        val_loss = total_val_loss / len(val_ds)
        print(f"Epoch {epoch+1} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

        scheduler.step()

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), save_path)
            print(f" ▶️Model saved to {save_path} (val_loss={val_loss:.4f})")
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"\u23f9 Early stopping at epoch {epoch+1}")
                break

    return model


In [10]:

crop_root = "disparity_crops"
annot_root = "train_annotations"
distance_json_path = "distance_estimates_filtered.json"

dataset = ModeAndFeatureDataset(
    crop_root=crop_root,
    annot_root=annot_root,
    distance_json_path=distance_json_path,
    max_items=10000
)


model = train_extended_model(dataset, save_path="420_3.pth")


[Train 1]: 100%|██████████| 125/125 [00:00<00:00, 193.41it/s]


Epoch 1 | Train Loss: 0.4203 | Val Loss: 0.3741
 ▶️Model saved to 420_3.pth (val_loss=0.3741)


[Train 2]: 100%|██████████| 125/125 [00:00<00:00, 185.58it/s]


Epoch 2 | Train Loss: 0.3241 | Val Loss: 0.3441
 ▶️Model saved to 420_3.pth (val_loss=0.3441)


[Train 3]: 100%|██████████| 125/125 [00:00<00:00, 190.17it/s]


...

Epoch 248 | Train Loss: 0.1584 | Val Loss: 0.2144


[Train 249]: 100%|██████████| 125/125 [00:00<00:00, 187.29it/s]


Epoch 249 | Train Loss: 0.1519 | Val Loss: 0.1718


[Train 250]: 100%|██████████| 125/125 [00:00<00:00, 186.73it/s]


Epoch 250 | Train Loss: 0.1490 | Val Loss: 0.2324


In [13]:
import os
import json
import numpy as np
from tqdm import tqdm
from PIL import Image
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

# ----- 設定 -----
MODEL_PATH = "420_3.pth"
CROP_ROOT = "outputs/test_crops"
ANNOT_ROOT = "test_annotations"
DISTANCE_JSON = "outputs/testdistance_estimates_filtered.json"
OUTPUT_JSON = "submission.json"

# ----- データセット定義 -----
class ExtendedInferenceDataset(Dataset):
    def __init__(self, crop_root, annot_root, distance_json_path):
        self.items = []
        self.distances = {}
        if distance_json_path:
            with open(distance_json_path, encoding='utf-8') as f:
                self.distances = json.load(f)

        scene_ids = sorted(os.listdir(crop_root))
        for sid in scene_ids:
            if not sid.isdigit():
                continue
            crop_dir = os.path.join(crop_root, sid)
            annot_path = os.path.join(annot_root, f"{sid}.json")
            if not os.path.exists(annot_path): continue

            files = sorted([f for f in os.listdir(crop_dir) if f.endswith(".png")])
            with open(annot_path, encoding='utf-8') as f:
                ann = json.load(f)
            min_len = min(len(files), len(ann['sequence']))
            if min_len < 15: continue

            own_speeds = np.array([ann['sequence'][i]['OwnSpeed'] / 3.6 for i in range(min_len)], dtype=np.float32)
            angles = np.array([ann['sequence'][i]['StrDeg'] for i in range(min_len)], dtype=np.float32)
            dmap = self.distances.get(sid, {})
            dists_all = []
            for i in range(min_len):
                try:
                    val = float(dmap.get(str(i), 0.0))
                    if not np.isfinite(val): val = 0.0
                except:
                    val = 0.0
                dists_all.append(val)
            dists_all = np.array(dists_all, dtype=np.float32)

            def smooth(x, k):
                return np.convolve(x, np.ones(k)/k, mode='same')

            for i in range(min_len - 14):
                d = dists_all[i:i+15]
                s = own_speeds[i:i+15]
                a = angles[i:i+15]
                d1 = np.gradient(d)
                d2 = np.gradient(d1)
                s1 = np.gradient(s)
                rel_acc = d2 - np.mean(d2)

                d_smooths = []
                for w in [3, 5, 7, 11]:
                    smoothed = smooth(d, w)[:15]
                    d_smooths.append(smoothed)
                    d_smooths.append(np.gradient(smoothed))

                img_paths = [os.path.join(crop_dir, files[j]) for j in range(i, i + 15)]
                modes = []
                for p in img_paths:
                    img = np.array(Image.open(p).convert("L")).flatten()
                    if img.size == 0:
                        img_mode = 0.0
                    else:
                        vals, counts = np.unique(img, return_counts=True)
                        img_mode = float(vals[np.argmax(counts)]) / 255.0
                    modes.append(img_mode)

                feature = np.concatenate([modes, d, s, a, s1, d1, d2, rel_acc] + d_smooths)
                own_speed = float(np.mean(s))
                self.items.append((torch.tensor(feature, dtype=torch.float32), own_speed, sid))

    def __len__(self): return len(self.items)
    def __getitem__(self, idx): return self.items[idx]

# ----- モデル定義（BatchNormあり）-----
class ExtendedFeatureModel(nn.Module):
    def __init__(self, in_dim=225):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(in_dim, 512), nn.BatchNorm1d(512), nn.ReLU(),
            nn.Linear(512, 256), nn.BatchNorm1d(256), nn.ReLU(),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64), nn.BatchNorm1d(64), nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.fc(x).squeeze(1)

# ----- 推論 -----
dataset = ExtendedInferenceDataset(CROP_ROOT, ANNOT_ROOT, DISTANCE_JSON)
loader = DataLoader(dataset, batch_size=32, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ExtendedFeatureModel(in_dim=dataset[0][0].shape[0]).to(device)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.eval()

predictions = {}
with torch.no_grad():
    for feats, own_speeds, sids in tqdm(loader, desc="Predicting"):
        feats = feats.to(device)
        own_speeds = own_speeds.to(device)
        pred_rel = model(feats)
        pred_abs = pred_rel + own_speeds  # 相対速度 + 自車速度 = 先行車速度
        pred_abs = pred_abs.cpu().numpy()

        for p, sid in zip(pred_abs, sids):
            if sid not in predictions:
                predictions[sid] = []
            predictions[sid].append(float(p * 3.6))  # m/s → km/h

# ----- 前の値で補完して保存 -----
formatted_output = {}
for sid_file in sorted(os.listdir(ANNOT_ROOT)):
    if not sid_file.endswith(".json") or not sid_file[:-5].isdigit():
        continue
    sid = sid_file[:-5]
    with open(os.path.join(ANNOT_ROOT, sid_file), encoding='utf-8') as f:
        ann = json.load(f)
    total_frames = len(ann["sequence"])
    preds = predictions.get(sid, [])
    padded = []
    for i in range(total_frames):
        if i < len(preds):
            padded.append(preds[i])
        else:
            padded.append(padded[-1] if padded else 0.0)  # 前の値で補完
    formatted_output[sid] = padded

with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(formatted_output, f, indent=2)
print(f"✅ Saved: {OUTPUT_JSON}")


Predicting: 100%|██████████| 754/754 [00:00<00:00, 1286.10it/s]


✅ Saved: submission.json
